# ✈️ Flight Delay Prediction
## 01 · Data Collection
Predict **before departure** whether a US domestic flight will arrive **more than 15 minutes late**.

**Pipeline:** 01 data collection → 02 data understanding & cleaning → 03 EDA & feature engineering → 04 model selection → 05 hyper-parameter tuning → 06 final evaluation & export

**Reads:** `data/archive.zip` (Kaggle *2015 Flight Delays and Cancellations*) *or* the csv files already placed in `data/raw/`  
**Writes:** `data/raw/flights.csv`, `airlines.csv`, `airports.csv`

In [ ]:
import zipfile
import pandas as pd

from pathlib import Path

# Repo layout (notebooks live in <repo>/notebooks/, everything else is resolved from the repo root):
#   data/raw        <- original Kaggle csv files (flights.csv, airlines.csv, airports.csv)
#   data/processed  <- intermediate files passed from one notebook to the next
#   model/          <- saved models / configs / final bundle
#   outputs/        <- small reports (csv tables)
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_RAW = ROOT / "data" / "raw"
DATA_PROC = ROOT / "data" / "processed"
MODEL_DIR = ROOT / "model"
OUT_DIR = ROOT / "outputs"
for d in (DATA_RAW, DATA_PROC, MODEL_DIR, OUT_DIR):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42


### Extract the dataset (skipped if the csv files are already in `data/raw/`)

In [ ]:
REQUIRED = ["flights.csv", "airlines.csv", "airports.csv"]
missing = [f for f in REQUIRED if not (DATA_RAW / f).exists()]

if missing:
    candidates = [ROOT / "data" / "archive.zip", ROOT / "archive.zip", Path("/content/archive.zip")]
    zip_path = next((p for p in candidates if p.exists()), None)
    if zip_path is None:
        raise FileNotFoundError(f"Missing {missing}. Put the 3 Kaggle csv files in {DATA_RAW} "
                                f"or place archive.zip in {ROOT / 'data'}")
    with zipfile.ZipFile(zip_path, "r") as zf:
        zf.extractall(DATA_RAW)
    print("Extraction completed from", zip_path)
else:
    print("All raw files already present in", DATA_RAW)

print({f: f"{(DATA_RAW / f).stat().st_size / 1e6:.1f} MB" for f in REQUIRED})

### Load data (only columns that are known before departure + target)

In [ ]:
# Load data (only columns that are known before departure + target)
cols = ["YEAR", "MONTH", "DAY", "DAY_OF_WEEK", "AIRLINE", "FLIGHT_NUMBER", "ORIGIN_AIRPORT", "DESTINATION_AIRPORT",
        "SCHEDULED_DEPARTURE", "SCHEDULED_TIME", "DISTANCE", "SCHEDULED_ARRIVAL", "ARRIVAL_DELAY", "DIVERTED", "CANCELLED"]
df = pd.read_csv(f"{DATA_RAW}/flights.csv", usecols=cols,
                 dtype={"AIRLINE": str, "ORIGIN_AIRPORT": str, "DESTINATION_AIRPORT": str})
airlines = pd.read_csv(f"{DATA_RAW}/airlines.csv")
airports = pd.read_csv(f"{DATA_RAW}/airports.csv")
print(df.shape)
df.head()